# Célula 1: Preparação e Vocabulário (A Base)
O primeiro passo de qualquer rede de PLN moderna é representar o texto através de Embeddings
. Vamos criar nosso vocabulário de 20 palavras.

Este diagrama ilustra a transformação do ID numérico de uma palavra em um vetor denso no espaço de embeddings.
```mermaid
graph LR
    A[Entrada: ID numérico] -->|Ex: 'cachorro' = 2| B(Camada de Embedding)
    B -->|nn.Embedding 20x4| C[Saída: Vetor denso]
    C -->|Tamanho: 4 números| D((Ex: 0.1, -0.5, 0.8, 0.2))

    style B fill:#f9f,stroke:#333,stroke-width:2px
```

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Vocabulário didático de 20 palavras
vocabulario = [
    "o", "a", "cachorro", "gato", "comeu", "dormiu", "feliz", "triste",
    "muito", "pouco", "rápido", "lento", "hoje", "ontem", "correu",
    "parou", "bom", "ruim", "filme", "livro"
]

# Dicionário para converter palavra em número (ID)
word2idx = {word: idx for idx, word in enumerate(vocabulario)}
vocab_size = len(vocabulario)
embedding_dim = 4 # Cada palavra será um vetor de 4 números

# Criando a camada de Embedding
embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

# Testando com a palavra "cachorro"
id_cachorro = torch.tensor([word2idx["cachorro"]])
vetor_cachorro = embedding_layer(id_cachorro)
print(f"ID da palavra 'cachorro': {id_cachorro.item()}")
print(f"Vetor Embedding de 'cachorro': {vetor_cachorro.detach().numpy()}")

ID da palavra 'cachorro': 2
Vetor Embedding de 'cachorro': [[ 1.2087839   0.22056904 -0.6898761  -0.42141876]]


# Célula 2: Nível Básico (O Neurônio e Camada Linear)
Aqui, aplicamos uma transformação linear (pesos e vieses) seguida de uma função de ativação, que é a base da nossa rede neural.

Aqui representamos a menor unidade de aprendizado: uma transformação linear que multiplica os pesos e soma o viés, seguida de uma ativação não linear (ReLU)

```mermaid
graph LR
    A[Vetor Embedding] -->|Tamanho: 4| B(Camada Linear)
    B -->|Pesos W e Viés b <br> nn.Linear| C[Saída Linear]
    C -->|Tamanho: 8| D(Função de Ativação)
    D -->|nn.ReLU| E[Saída Ativada]

    style B fill:#bbf,stroke:#333,stroke-width:2px
    style D fill:#f96,stroke:#333,stroke-width:2px
```

In [2]:
# 2. Camada Linear Simples + Ativação
hidden_size = 8

# A camada linear mapeia o vetor de tamanho 4 para um tamanho oculto de 8
camada_linear = nn.Linear(in_features=embedding_dim, out_features=hidden_size)
ativacao_relu = nn.ReLU() # Adiciona não-linearidade

# Passando o vetor do "cachorro" pela camada linear
saida_linear = camada_linear(vetor_cachorro)
saida_ativada = ativacao_relu(saida_linear)

print("Saída após a Camada Linear e ReLU:")
print(saida_ativada.detach().numpy())

Saída após a Camada Linear e ReLU:
[[0.32212073 0.         0.         0.         0.         0.5251075
  0.         0.        ]]


# Célula 3: Nível Intermediário (Rede Feedforward / MLP)
As MLPs processam uma janela fixa de palavras para tarefas como classificação
. Vamos simular a frase "o cachorro correu" prevendo se o sentimento é positivo, negativo ou neutro (3 classes).

Este diagrama mostra uma arquitetura que recebe uma janela de contexto fixo de palavras e as processa em múltiplas camadas para realizar uma classificação de texto (como uma análise de sentimentos)

```mermaid
graph TD
    A[Entrada: 3 IDs de palavras] -->|Ex: 'o', 'cachorro', 'correu'| B(Camada de Embeddings)
    B -->|3 vetores de tamanho 4| C[Concatenação / Achatar]
    C -->|Vetor único de tamanho 12| D(Camada Oculta 1)
    D -->|Linear 12 -> 8| E(Ativação ReLU)
    E -->|Vetor de tamanho 8| F(Camada de Saída)
    F -->|Linear 8 -> 3| G[Saída: 3 Classes]
    G -->|Positivo, Negativo, Neutro| H((Predição Final))

    style B fill:#f9f,stroke:#333,stroke-width:2px
    style D fill:#bbf,stroke:#333,stroke-width:2px
    style E fill:#f96,stroke:#333,stroke-width:2px
    style F fill:#bbf,stroke:#333,stroke-width:2px
```

In [3]:
# 3. Rede Feedforward (MLP)
class ClassificadorMLP(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        # Multiplicamos o embed_dim por 3 pois nossa janela fixa será de 3 palavras
        self.fc1 = nn.Linear(embed_dim * 3, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x tem 3 palavras. O embedding converte cada uma em vetor
        embeds = self.embedding(x)
        # Achata os 3 vetores em um grande vetor contínuo
        embeds_achato = embeds.view(1, -1)
        oculto = self.relu(self.fc1(embeds_achato))
        saida = self.fc2(oculto)
        return saida

# Instanciando o modelo para 3 classes de sentimento
modelo_mlp = ClassificadorMLP(vocab_size, embedding_dim, hidden_size, num_classes=3)

# Frase: "o cachorro correu"
frase_janela = torch.tensor([word2idx["o"], word2idx["cachorro"], word2idx["correu"]])
predicao_mlp = modelo_mlp(frase_janela)

print("Predição (Scores brutos para Positivo, Negativo, Neutro):")
print(predicao_mlp.detach().numpy())

Predição (Scores brutos para Positivo, Negativo, Neutro):
[[-0.01511033 -0.06691904 -0.21455754]]


# Célula 4: Nível Avançado (Rede Neural Recorrente - RNN)
As RNNs resolvem o problema de janelas fixas mantendo um estado oculto (memória) à medida que processam sequências de tamanho variável
. Aqui está o código para um Modelo de Linguagem prevendo a próxima palavra

Aqui vemos como a RNN lida com sequências de tamanho variável. O detalhe principal é o loop do "Estado Oculto" (h
t
​
 ) que atua como a memória da rede a cada nova palavra inserida
```mermaid
graph TD
    A[Sequência de IDs] -->|Iteração passo a passo| B(Camada de Embeddings)
    B -->|Vetor da Palavra Atual| C(Camada RNN)

    %% Loop do Estado Oculto
    C -- "Atualiza Estado Oculto (Memória)" --> C

    C -->|Após ler toda a sequência| D[Último Estado Oculto]
    D -->|Vetor de tamanho 12| E(Camada Linear de Saída)
    E -->|Linear 12 -> 20 <br> nn.Linear| F[Distribuição de Probabilidade]
    F --> G((Predição da Próxima Palavra))

    style C fill:#8f8,stroke:#333,stroke-width:2px
    style E fill:#bbf,stroke:#333,stroke-width:2px
```

In [4]:
# 4. Rede Neural Recorrente (RNN) para Modelagem de Linguagem
class RNNLinguagem(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        # RNN reaproveita pesos a cada passo temporal
        self.rnn = nn.RNN(input_size=embed_dim, hidden_size=hidden_size, batch_first=True)
        # Camada final para prever qual das 20 palavras é a próxima
        self.fc_out = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        embeds = self.embedding(x)
        # A RNN retorna a saída e o último estado oculto (memória)
        saida_rnn, estado_oculto = self.rnn(embeds)
        # Queremos prever a próxima palavra com base no último passo da sequência
        predicao_palavra = self.fc_out(saida_rnn[:, -1, :])
        return predicao_palavra

modelo_rnn = RNNLinguagem(vocab_size, embedding_dim, hidden_size=12)

# Frase de tamanho variável: "o gato dormiu muito" (4 palavras)
frase_seq = torch.tensor([[word2idx["o"], word2idx["gato"], word2idx["dormiu"], word2idx["muito"]]])
predicao_prox_palavra = modelo_rnn(frase_seq)

print("Tamanho do vetor de predição (deve ser 20, uma chance para cada palavra):")
print(predicao_prox_palavra.shape)

Tamanho do vetor de predição (deve ser 20, uma chance para cada palavra):
torch.Size([1, 20])


# Célula 5: O Guia Prático de Treinamento (Loss e Optimizer)
Para a rede aprender, precisamos calcular o erro (Cross-Entropy Loss) e ajustar os pesos usando um otimizador.

Por fim, este diagrama não é apenas da arquitetura, mas do grafo de computação (ciclo de treinamento). Ele ilustra o fluxo de passagem dos dados (forward), o cálculo do erro, e a propagação de volta (backpropagation) para o otimizador ajustar os pesos.

```mermaid
graph LR
    A[Dados de Entrada] --> B(Forward Pass)
    B -->|Predição da Rede| C{Cálculo do Erro}
    C -->|Cross-Entropy Loss| D(Backward Pass)
    D -->|Cálculo dos Gradientes| E(Otimizador)
    E -->|Ajuste de Pesos e Viés| F((Fim da Época))
    F -.->|Repete o ciclo| A

    style B fill:#ddd,stroke:#333,stroke-width:2px
    style C fill:#f66,stroke:#333,stroke-width:2px
    style D fill:#fd9,stroke:#333,stroke-width:2px
    style E fill:#9df,stroke:#333,stroke-width:2px
```


In [5]:
# 5. Fluxo Básico de Treinamento
criterio_perda = nn.CrossEntropyLoss() # Compara previsão com a palavra real
otimizador = optim.Adam(modelo_rnn.parameters(), lr=0.01) # Ajusta hiperparâmetro de taxa de aprendizado

# Simulação: Prever a palavra "hoje" após "o gato dormiu muito"
alvo_real = torch.tensor([word2idx["hoje"]])

for epoca in range(5):
    otimizador.zero_grad() # Limpa os gradientes antigos

    # 1. Faz a previsão (Forward)
    previsao = modelo_rnn(frase_seq)

    # 2. Calcula o Erro (Loss)
    perda = criterio_perda(previsao, alvo_real)

    # 3. Calcula os ajustes necessários (Backward) e atualiza pesos (Step)
    perda.backward()
    otimizador.step()

    print(f"Época {epoca+1} - Erro (Loss): {perda.item():.4f}")

Época 1 - Erro (Loss): 3.0036
Época 2 - Erro (Loss): 2.8151
Época 3 - Erro (Loss): 2.6382
Época 4 - Erro (Loss): 2.4670
Época 5 - Erro (Loss): 2.2929


## O Exercício Prático: Predição da Próxima Palavra
- Neste exercício, vamos treinar o modelo alimentando-o com sentenças. Em cada passo, a rede tentará prever a próxima palavra da sentença, e nós calcularemos o erro (Loss) comparando a previsão com a palavra que realmente veio a seguir.
- Como o processamento do corpus inteiro de uma só vez seria muito custoso para a memória, usaremos o método Stochastic Gradient Descent (SGD), calculando a perda para um pequeno lote (ou seja, cada uma das nossas 40 sentenças), ajustando os pesos e repetindo o processo.

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Preparação do nosso "Documento de 40 Sentenças"
# Em um cenário real, estas seriam extraídas de um texto.
# Aqui, simularemos 40 sentenças simples de 5 palavras para manter a didática.
vocabulario = ["o", "gato", "comeu", "peixe", "dormiu", "cachorro", "feliz", "<END>"]
word2idx = {word: idx for idx, word in enumerate(vocabulario)}
vocab_size = len(vocabulario)

# Simulando o documento: 40 sentenças idênticas/semelhantes para o modelo decorar o padrão
# Ex: "o gato comeu o peixe <END>"
sentenca_exemplo = [word2idx[w] for w in ["o", "gato", "comeu", "peixe", "<END>"]]
documento_40_sentencas = [sentenca_exemplo for _ in range(40)]

# 2. Definindo a Arquitetura da RNN
class RNNLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_size, batch_first=True)
        self.fc_out = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        embeds = self.embedding(x)
        # A RNN passa por toda a sequência. saida_rnn contém o estado de CADA passo temporal
        saida_rnn, _ = self.rnn(embeds)
        # Convertendo os estados ocultos em distribuições de probabilidade para prever o vocabulário
        predicoes = self.fc_out(saida_rnn)
        return predicoes

modelo = RNNLanguageModel(vocab_size, embed_dim=8, hidden_size=16)

# 3. Otimizador e Função de Custo
criterio_perda = nn.CrossEntropyLoss() # Calcula a entropia cruzada entre a previsão e a palavra real [2]
otimizador = optim.SGD(modelo.parameters(), lr=0.1) # Stochastic Gradient Descent [3]

# 4. Loop de Treinamento
print("Iniciando Treinamento...")
for epoca in range(5):
    perda_total = 0

    # Iteramos sobre cada uma das 40 sentenças do nosso documento
    for sentenca in documento_40_sentencas:
        otimizador.zero_grad()

        # A entrada são todas as palavras, exceto a última.
        # O alvo (target) são todas as palavras, deslocadas um passo para o futuro (exceto a primeira) [2].
        # Ex: Entrada = ["o", "gato", "comeu", "peixe"] | Alvo = ["gato", "comeu", "peixe", "<END>"]
        entrada = torch.tensor([sentenca[:-1]])
        alvo_real = torch.tensor([sentenca[1:]])

        # Faz o Forward Pass (prevê todas as próximas palavras de uma vez)
        predicoes = modelo(entrada)

        # Achata os tensores para calcular o erro
        predicoes_achatadas = predicoes.view(-1, vocab_size)
        alvo_achatado = alvo_real.view(-1)

        # Calcula a Perda e Atualiza os Pesos
        perda = criterio_perda(predicoes_achatadas, alvo_achatado)
        perda.backward() # Backpropagation [4]
        otimizador.step() # Atualiza os pesos [3]

        perda_total += perda.item()

    print(f"Época {epoca+1} - Perda Média: {perda_total/40:.4f}")

Iniciando Treinamento...
Época 1 - Perda Média: 0.7376
Época 2 - Perda Média: 0.1008
Época 3 - Perda Média: 0.0416
Época 4 - Perda Média: 0.0256
Época 5 - Perda Média: 0.0184


## Passo 1:
    Entradas e Alvos Deslocados Para treinar um modelo de linguagem, nós alimentamos a rede com uma frase e pedimos que ela preveja a sequência deslocada um passo para o futuro. Se a frase for "o gato comeu peixe", a rede usa "o" para tentar adivinhar "gato"; depois usa o contexto de "o gato" para adivinhar "comeu", e assim por diante.
## Passo 2:
    A RNN no Forward Pass A camada RNN processa as palavras sequencialmente. Diferente da rede linear das aulas anteriores, a RNN cospe uma previsão e também guarda um estado oculto em sua "memória", combinando-o com a próxima palavra que vai ler.
## Passo 3:
    A Função de Erro (Loss) Usamos a função CrossEntropyLoss. Em cada passo no tempo, ela compara a distribuição de probabilidade gerada pela rede com a próxima palavra real do corpus. A média de erro de toda a sentença é usada pelo otimizador (SGD) para ajustar os pesos da rede antes de ir para a próxima frase do lote de 40.
## Passo Extra:
    Geração de Texto (Após o treino) Como a gente usa esse modelo pronto? Aplicamos a amostragem repetida (repeated sampling). Nós fornecemos uma palavra inicial, pedimos para a rede prever a próxima, e usamos essa nova palavra gerada como a entrada para o passo seguinte, criando textos longos. Como a RNN pode ser treinada em qualquer tipo de texto, ela aprenderá o estilo específico daquele documento (seja literatura ou receitas culinárias).